# 09 — Evaluation: how good is the pipeline, with numbers?

> **Restart kernel + Run All.** After editing any `.py` file: restart the kernel,
> then Run All (no `%autoreload` — reloading leaves two copies of the same
> Pydantic class alive).

> **Cost gate:** everything here runs on committed files (`data/samples`,
> `data/eval/live_snapshot.json`, `data/fixtures`) — zero network by default.
> With `RUN_LIVE_LLM=1`, sections A and B spend real Gemini calls (grading the
> real extractor; **~4 calls cover the 20-entry snapshot** thanks to batching).

Six sections, each answering one question:

- **A — documents**: does the reader+OCR keep the text (`reader text recall`),
  and how accurate is the structured extraction vs the hand-written ground
  truth (`document grading`);
- **B — ingestion**: on a frozen snapshot of 20 REAL feed entries, how many
  state an explicit deadline (`rule deadline`), and what does the batched
  extractor do with them (`batched:` call math, `snapshot gate:` counts);
- **C — batching**: the free-tier fix is shown by B's call math — one Gemini
  call per `extraction_batch_size` entries instead of one per entry;
- **D — matching**: hard-filter drop reasons, score spreads and a threshold
  sweep over 32 fixtures x 5 profiles, plus an honest note on the scam labels;
- **E — notifications**: a simulated week proves the daily cap, quiet hours
  and "never twice for the same opportunity";
- **F — live rerun**: what to set to turn every number into a real measurement.

> **Read the provider line first.** Offline (`LLM_PROVIDER=fake`) the grading
> harness runs on **canned** fake output: the numbers then measure the harness,
> not the real extractor. Re-run with `RUN_LIVE_LLM=1` for real accuracy.


In [ ]:
import json
import os
import sys
from collections import Counter
from datetime import date, datetime, timedelta
from pathlib import Path

# Make oppradar_ai importable whether the kernel starts in ai/ or ai/notebooks/.
AI_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(AI_ROOT) not in sys.path:
    sys.path.insert(0, str(AI_ROOT))

import pandas as pd
from IPython.display import display

from oppradar_ai.config import Settings, get_settings, setup_logging
from oppradar_ai.documents import grade_extraction, normalize, read_document
from oppradar_ai.documents.extractors import (
    extract_certificate,
    extract_cv,
    extract_portfolio,
)
from oppradar_ai.ingestion.batch import BatchInput, extract_records_batch
from oppradar_ai.ingestion.budget import RunBudget
from oppradar_ai.ingestion.extract import deadline_line, find_deadline
from oppradar_ai.llm import get_stats
from oppradar_ai.matching import (
    apply_hard_filters,
    decide_notifications,
    prefs_from_settings,
    score_user,
)
from oppradar_ai.schemas import NotificationMode, Opportunity, Profile

# -- Cost gate: offline unless you explicitly opt in --------------------------
RUN_LIVE_LLM = Settings().run_live_llm
os.environ.pop('LLM_PROVIDER', None)      # drop the force of a previous run
_configured = Settings().llm_provider     # fresh read of ai/.env (get_settings is cached)
_can_live = _configured != 'gemini' or bool(Settings().gemini_key_value)
os.environ['LLM_PROVIDER'] = _configured if RUN_LIVE_LLM and _can_live else 'fake'
get_settings.cache_clear()

setup_logging('WARNING')
settings = get_settings().ensure_dirs()

print(f"provider: {settings.llm_provider}, embedding: {settings.embedding_provider}, "
      f"live LLM: {'yes' if RUN_LIVE_LLM and _can_live else 'no'}")

SAMPLES = AI_ROOT / 'data' / 'samples'
GROUND_TRUTH = SAMPLES / 'ground_truth'
EVAL = AI_ROOT / 'data' / 'eval'
FIXTURES = AI_ROOT / 'data' / 'fixtures'


## A — Document intake: text recall, then extraction accuracy

`reader text recall` checks, per document, whether the ground-truth name /
e-mail / country survive PDF text extraction **or OCR** (the scanned CV) —
purely local, no model. `document grading` runs the full extraction route for
the active provider and scores it field by field against
`data/samples/ground_truth/*.json` (case-insensitive, whitespace-normalized).


In [ ]:
DOCS = [
    ('cv_amine.pdf', 'cv_amine.json', extract_cv),
    ('cv_sara_scanned.pdf', 'cv_sara_scanned.json', extract_cv),
    ('portfolio_sara.docx', 'portfolio_sara.json', extract_portfolio),
    ('certificate_alx_amine.pdf', 'certificate_alx_amine.json', extract_certificate),
]
TEXT_FIELDS = ('full_name', 'email', 'country')  # scalar truth fields a reader can preserve

rows, matched, checked_total = [], 0, 0
for doc_name, truth_name, _ in DOCS:
    truth = json.loads((GROUND_TRUTH / truth_name).read_text(encoding='utf-8'))
    text = normalize(read_document(SAMPLES / doc_name).text)
    found = {field: normalize(truth[field]) in text for field in TEXT_FIELDS if truth.get(field)}
    if not found:
        continue
    matched += sum(1 for ok in found.values() if ok)
    checked_total += len(found)
    rows.append({'document': doc_name, **{f'{k} found': ok for k, ok in found.items()}})
display(pd.DataFrame(rows))
print(f"reader text recall: {matched}/{checked_total} ground-truth fields survive the "
      f"PDF/DOCX/OCR text layer (scanned CV exercises rapidocr)")


In [ ]:
grades = []
for doc_name, truth_name, extractor in DOCS:
    truth = json.loads((GROUND_TRUTH / truth_name).read_text(encoding='utf-8'))
    text = read_document(SAMPLES / doc_name).text
    grade = grade_extraction(doc_name, extractor(text, doc_name), truth)
    grades.append(grade)

display(pd.DataFrame([
    {'document': g.document, 'fields': g.fields_checked, 'matched': g.fields_matched,
     'accuracy': round(g.accuracy, 2)}
    for g in grades
]))
overall = sum(g.accuracy for g in grades) / len(grades)
print(f"document grading: overall accuracy={overall:.2f} across {len(grades)} documents "
      f"({sum(g.fields_checked for g in grades)} fields checked)")
if settings.llm_provider == 'fake':
    print("  ^ fake provider: these numbers grade the OFFLINE HARNESS (canned output), "
          "not the real extractor — re-run with RUN_LIVE_LLM=1 for real accuracy")


## B — Ingestion on the frozen snapshot of REAL feed entries

`data/eval/live_snapshot.json` froze 20 real entries (2026-10-22,
`scripts/snapshot_live.py`). First a **provider-independent** measurement: how
many entries state their deadline as a `Deadline:`/`Date limite:` line (the
rule the extractor trusts above any model). Then the batched extraction itself.


In [ ]:
snapshot = json.loads((EVAL / 'live_snapshot.json').read_text(encoding='utf-8'))
entries = snapshot['entries']

def entry_text(entry):
    return f"{entry['title']}\n{entry['summary']}"

explicit = sum(1 for e in entries if deadline_line(entry_text(e)))
any_date = sum(1 for e in entries if find_deadline(entry_text(e)))
print(f"rule deadline: {explicit}/{len(entries)} live entries state an explicit 'Deadline:' "
      f"line; {any_date} carry any parseable date at all")

labels_path = EVAL / 'live_labels.json'
if labels_path.exists():
    labels = json.loads(labels_path.read_text(encoding='utf-8'))
    print(f"gate labels: {sum(1 for v in labels.values() if v)} real opportunities / "
          f"{sum(1 for v in labels.values() if not v)} non-opportunities, by hand")
else:
    print("gate labels: not filled yet — copy live_labels.template.json to "
          "live_labels.json and label the 20 entries by hand to score the gate")


In [ ]:
batch_inputs = [
    BatchInput(text=entry_text(e), url=e['url'], source_name=e['source'], title_hint=e['title'])
    for e in entries
]
budget = RunBudget(settings)
outcomes = extract_records_batch(batch_inputs, budget=budget, settings=settings)

accepted = [o.opportunity for o in outcomes if o.opportunity]
rejected = [o for o in outcomes if o.rejected_reason]
pending = [o for o in outcomes if o.pending]
print(f"batched: {budget.extractions.made + budget.extractions.cached} LLM calls cover "
      f"{len(entries)} entries (extraction_batch_size={settings.extraction_batch_size}; "
      f"unbatched would need {len(entries)} separate calls)")
print(f"snapshot gate: {len(accepted)} opportunities, {len(rejected)} rejected "
      f"(listicles/news), {len(pending)} pending (budget/quota — retried next run)")
if settings.llm_provider == 'fake':
    print("  ^ fake provider: the fake extractor cycles canned items, so the "
          "gate counts demonstrate the MECHANICS; real numbers need RUN_LIVE_LLM=1")


## D — Matching: filters, score spread, threshold sweep

32 committed fixtures x 5 profiles, with `today` pinned to 2026-01-01 so the
metrics never drift with the calendar. The sweep columns answer the one
question the backend cares about: *where should NOTIFY_THRESHOLD sit?*


In [ ]:
opps = [Opportunity.model_validate(row)
        for row in json.loads((FIXTURES / 'opportunities.json').read_text(encoding='utf-8'))]
profiles = [Profile.model_validate(row)
            for row in json.loads((FIXTURES / 'profiles.json').read_text(encoding='utf-8'))]
today = date(2026, 1, 1)  # pinned: these metrics must be reproducible forever

rows, reasons = [], Counter()
for profile in profiles:
    outcome = apply_hard_filters(profile, opps, today=today)
    reasons.update(d.reason.value for d in outcome.dropped)
    matches = score_user(profile, opps, settings=settings, today=today)
    scores = [m.score for m in matches]
    rows.append({
        'profile': profile.user_id, 'passed': len(outcome.passed), 'dropped': len(outcome.dropped),
        'min': round(min(scores), 1) if scores else None,
        'median': round(sorted(scores)[len(scores) // 2], 1) if scores else None,
        'max': round(max(scores), 1) if scores else None,
        **{f'>={t}': sum(1 for s in scores if s >= t) for t in (60, 70, 80, 90)},
    })
display(pd.DataFrame(rows))
print("matching: hard-filter drop reasons across all profiles:", dict(reasons))
print("threshold sweep: the >=60/>=70/>=80/>=90 columns count matches that would be "
      "notified per profile at that NOTIFY_THRESHOLD")

scam_labels = json.loads((EVAL / 'scam_labels.json').read_text(encoding='utf-8'))
print(f"scam labels: {sum(1 for v in scam_labels.values() if not v)} safe / "
      f"{sum(1 for v in scam_labels.values() if v)} scam, hand-labelled on the frozen "
      "snapshot — the rule detector that would auto-flag them is NOT built yet "
      "(rows only drop when trust_level=scam or scam_flags are set)")


## E — Notifications: one simulated week

The same matches, decided once per day for seven days plus a 23:30 run, with
the history fed back exactly like the backend persists it. The three guarantees
come out as booleans: daily cap, quiet hours, never-twice.


In [ ]:
profile = profiles[0]
matches = score_user(profile, opps, settings=settings, today=today)
prefs = prefs_from_settings(profile.user_id, settings)
if prefs.mode is not NotificationMode.INSTANT:
    print(f"note: digest_mode={settings.digest_mode} applies in production; "
          "the sim below uses instant delivery to exercise the cap")
    prefs = prefs.model_copy(update={'mode': NotificationMode.INSTANT})
if not any(m.score >= prefs.threshold for m in matches):
    print(f"note: no match reaches NOTIFY_THRESHOLD={prefs.threshold} with this "
          "embedder — the sim below runs at threshold 0")
    prefs = prefs.model_copy(update={'threshold': 0.0})

history: list = []
sent_per_day = []
for day in range(7):
    now = datetime(2026, 1, 5, 9, 0) + timedelta(days=day)
    rows = decide_notifications(matches, prefs, history, now=now)
    history.extend(rows)
    sent_per_day.append(sum(1 for r in rows if r.status.value == 'sent'))

night = decide_notifications(matches, prefs, [], now=datetime(2026, 1, 6, 23, 30))
sent_keys = [r.opportunity_id for r in history if r.status.value == 'sent']
print(f"notifications week: sent per day {sent_per_day} with cap {prefs.daily_cap} — "
      f"cap honored: {all(s <= prefs.daily_cap for s in sent_per_day)}")
print(f"quiet hours respected: "
      f"{all(r.status.value != 'sent' for r in night)} (a 23:30 decision defers, never sends)")
print(f"no duplicate sends: {len(sent_keys) == len(set(sent_keys))} "
      f"({len(sent_keys)} sent rows, all distinct)")


## F — Turning every number into a real measurement

All sections re-run unchanged with real providers; only the provider line and
the numbers change. Sections A + B then spend a handful of Gemini calls
(batching keeps B to ~4 for the 20 entries); D + E are always local.


In [ ]:
stats = get_stats()
print(f"run live: set RUN_LIVE_LLM=1 in ai/.env, restart the kernel and Run All — "
      f"section A grades the real extractor, section B the real gate (~4 Gemini calls "
      f"for {len(entries)} entries thanks to batching)")
print(f"llm usage this session: calls={stats.calls}, cached={stats.cached}, "
      f"errors={stats.errors}, failed attempts={stats.failed_attempts}")


### Reading the numbers — known limitations

- **offline ≠ accuracy**: with the fake provider the harness is measured, not
  the extractor; only a live run gives real accuracy;
- **no relevance labels yet**: `Precision@5` / `nDCG@5` from `docs/ai-design.md`
  stay TBD until someone labels which fixture opportunities fit which profile —
  the threshold sweep is the honest stand-in;
- **scam detector missing**: the two hand-labelled scams in
  `scam_labels.json` drop only when `trust_level`/`scam_flags` are set — no
  rule detector sets them automatically yet;
- **OCR quality is model- and scan-dependent**: the single scanned CV is one
  data point; add harder scans to `data/samples/` to make section A meaningful;
- **snapshot frozen 2026-10-22**: refresh with
  `python scripts/snapshot_live.py` (needs a live RSS pull) to re-baseline B.
